# Week 2 — Agents: Constructing a Goal-Based Agent using an LLM

**CS F407 Artificial Intelligence · Lab 2**

An autonomous warehouse vehicle must find a collision-free path from the loading bay `S` to the dispatch area `G`, moving one grid square at a time (Up, Down, Left, Right). This notebook specifies the problem (Task 1), designs a goal-based agent (Task 2), and uses an LLM to implement it through iterative prompting (Task 3). It then tests the result and compares search strategies, including on larger warehouses (Think About It).

All answers are inline and are also collected in [`answers-to-questions/answers.md`](../answers-to-questions/answers.md). The code uses only the Python standard library.

| Section | Contents |
|---|---|
| Task 1 | Environment, goal, actions, state, goal-based vs reflex (Q1–5) |
| Think About It | A twice-as-large warehouse: scaling experiment across search strategies |
| Task 2 | Agent design and block diagram |
| Task 3 | Prompt, first LLM attempt, testing, improved prompt, final agent (Q1–4) |

In [1]:
import heapq
import random
import time
from collections import deque
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

WAREHOUSE_MAP = '''
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
'''

## Task 1 — Understanding the Problem

A quick look at the map before answering:

In [2]:
rows = WAREHOUSE_MAP.strip("\n").split("\n")
print(f"{len(rows)} rows x {len(rows[0])} columns; every row the same width: {len({len(r) for r in rows}) == 1}")
cells = [(r, c, ch) for r, row in enumerate(rows) for c, ch in enumerate(row)]
print("S at", [(r, c) for r, c, ch in cells if ch == "S"], "  G at", [(r, c) for r, c, ch in cells if ch == "G"])
print("free cells (., S, G):", sum(ch != "#" for _, _, ch in cells), "  obstacle cells:", sum(ch == "#" for _, _, ch in cells))
print("Manhattan distance S->G:", abs(1 - 1) + abs(19 - 1))

7 rows x 21 columns; every row the same width: True
S at [(1, 1)]   G at [(1, 19)]
free cells (., S, G): 66   obstacle cells: 81
Manhattan distance S->G: 18


**1. What is the environment?**
The warehouse floor: a 7 × 21 grid of cells, each either free (`.`, `S`, `G`) or an obstacle (`#`, a shelving unit or the outer wall). There are 66 free cells. The environment is:
- **fully observable:** the whole map is known in advance;
- **deterministic:** a move always goes exactly one square;
- **static:** the shelves do not move while the agent plans;
- **discrete:** a finite set of cells and actions;
- **single-agent:** no other vehicles;
- **known:** the effect of each action is known.

**2. What is the goal of the agent?**
To reach the dispatch cell `G` at (row 1, column 19), starting from `S` at (1, 1), along a path that never enters an obstacle cell. As a performance measure it is natural to prefer **shorter paths** (fewer moves means less time and energy), so the practical goal is a *shortest* collision-free path. The lab only requires "a collision-free path", so shortest is our added design choice.

**3. What actions are available?**
`Up`, `Down`, `Left`, `Right`, each moving one square:
- with positions written as (row, column), `Up` = (−1, 0), `Down` = (+1, 0), `Left` = (0, −1), `Right` = (0, +1);
- an action is only applicable if the target cell is free (not `#` and inside the grid).

Every action costs 1.

**4. What information must the agent maintain in order to choose its next action?**
- a **model of the environment**: the map, so it can tell which moves are legal (the transition model);
- its **current state**, i.e. its position (row, column);
- the **goal** (the position of `G`) and a goal test;
- while planning, the search's bookkeeping: the **frontier** of positions still to explore, the set of **visited/reached** positions (so it does not loop), and a **parent pointer** per reached position (so the path can be reconstructed);
- while executing, the **remaining plan**: the sequence of actions still to perform.

**5. Why is this a goal-based agent rather than a simple reflex agent?**
A simple reflex agent maps the *current percept* directly to an action with condition–action rules ("if the cell ahead is free, move right"). It has no notion of where it is trying to get to or of what its actions will lead to. In this map that fails: `S` and `G` are on the same row, but the direct route is blocked by the shelf at column 6. The right first move depends on the *whole* layout and on where `G` is, not on the local surroundings.

A goal-based agent has an explicit goal and a model of how actions change the state. It **looks ahead**: it simulates sequences of actions ("if I go down here, where do I end up?"), searches for one that reaches the goal, and then acts on it. If the goal changes (a different dispatch bay), the same agent works unchanged with a new goal; a reflex agent's rules would have to be rewritten.

## Task 2 — Designing the Agent

| Component | Design |
|---|---|
| **Environment** | `Warehouse`: the parsed grid, with `is_free(pos)`, `successors(pos)` (the transition model) and `step(pos, action)`, which *executes* a move and raises on collision |
| **Current state** | the vehicle's position `(row, col)`; initially the position of `S` |
| **Goal** | the position of `G`; goal test `state == goal` |
| **Available actions** | `ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}`, applicable when the target cell is free |
| **Decision-making component** | a search-based planner (breadth-first search) that returns an action sequence from the current state to the goal; the agent executes it one action at a time and replans if a move is ever blocked |

### Block diagram

```
                 ┌────────────────────────── GOAL-BASED AGENT ──────────────────────────┐
                 │                                                                      │
   percept       │   ┌──────────────┐    ┌──────────────────┐    ┌─────────────────┐     │
  (position, ───────▶│ State update │───▶│ Current state    │───▶│  Planner        │     │
   map)          │   │ (sensors)    │    │ (row, col)       │    │  (BFS search)   │     │
                 │   └──────────────┘    └──────────────────┘    │                 │     │
                 │                                               │ "What will      │     │
                 │   ┌──────────────┐    ┌──────────────────┐    │  happen if I    │     │
                 │   │ Goal         │───────────────────────────▶│  do action A?"  │     │
                 │   │ G = (1, 19)  │    │ Model: map +     │───▶│  "Does it reach │     │
                 │   └──────────────┘    │ transition rules │    │   the goal?"    │     │
                 │                       └──────────────────┘    └────────┬────────┘     │
                 │                                                        │ plan         │
                 │                                               ┌────────▼────────┐     │
                 │                                               │ Plan executor   │     │
                 │                                               │ (next action)   │     │
                 │                                               └────────┬────────┘     │
                 └────────────────────────────────────────────────────────┼──────────────┘
                                                                          │ action (Up/Down/Left/Right)
                                     ┌────────────────────────────────────▼──────────────┐
                                     │  ENVIRONMENT: warehouse grid (moves vehicle 1 cell)│
                                     └────────────────────────────────────┬──────────────┘
                                                                          │ new percept
                                                   (back to State update) ◀┘
```

The loop is the goal-based architecture from the lecture. The agent perceives its state, uses its model to predict the consequences of action sequences, chooses the sequence that achieves the goal, and acts. It keeps acting and perceiving until the goal test succeeds.

## Task 3 — Prompt Engineering

### First prompt

I started with the lab's suggested prompt, unchanged, pasting in the map:

> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem shown above. The program should
> - represent the warehouse as a two-dimensional grid;
> - determine a collision-free path from S to G;
> - avoid all obstacles;
> - print either the path found or a suitable message if no path exists;
> - explain the search algorithm that has been chosen and why it is appropriate.

### First LLM attempt (as returned)

The core of the returned program is below. Its module docstring and printing code are trimmed, but the logic is unchanged.

In [3]:
def solve_warehouse(grid_text):
    """Find a path from S to G using Breadth-First Search (BFS).

    BFS explores the grid level by level, so the first time it reaches G
    it has found a shortest path (every move costs the same). It is complete:
    if a path exists, BFS will find it.
    """
    grid = [list(row) for row in grid_text.strip().split("\n")]
    rows, cols = len(grid), len(grid[0])

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "S":
                start = (r, c)
            elif grid[r][c] == "G":
                goal = (r, c)

    moves = {"Up": (0, -1), "Down": (0, 1), "Left": (-1, 0), "Right": (1, 0)}

    queue = deque([(start, [])])
    visited = set()
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == goal:
            return path
        if (r, c) in visited:
            continue
        visited.add((r, c))
        for name, (dx, dy) in moves.items():
            nr, nc = r + dx, c + dy
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#":
                queue.append(((nr, nc), path + [name]))
    return None


first_path = solve_warehouse(WAREHOUSE_MAP)
print("Path found:" if first_path else "No path exists.", first_path)
print("Length:", len(first_path) if first_path else None)

Path found: ['Down', 'Down', 'Down', 'Down', 'Right', 'Down', 'Down', 'Left', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down']
Length: 20


It ran on the first attempt and printed a 20-move path. That looks like success. Before accepting it, I tested it against the specification rather than trusting the printout:

- **Replay test:** execute the printed actions from `S` using the lab's meaning of Up/Down/Left/Right, and check that every move lands on a free cell and the last one lands on `G`.
- **No-path test:** wall off `G`; it should report that no path exists.
- **Malformed-input tests:** a map with ragged rows, and a map with no `S`. These should give a clear error, not a crash deep inside the search.

In [4]:
ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}   # (d_row, d_col), per the lab

def replay(map_text, actions):
    # Execute actions from S with the lab's semantics. Returns (ok, message).
    grid = map_text.strip("\n").split("\n")
    pos = next((r, c) for r, row in enumerate(grid) for c, ch in enumerate(row) if ch == "S")
    for i, a in enumerate(actions):
        dr, dc = ACTIONS[a]
        pos = (pos[0] + dr, pos[1] + dc)
        if not (0 <= pos[0] < len(grid) and 0 <= pos[1] < len(grid[pos[0]])) or grid[pos[0]][pos[1]] == "#":
            return False, f"move {i + 1} ({a}) collides at {pos}"
    ok = grid[pos[0]][pos[1]] == "G"
    return ok, f"ends at {pos} ({'G' if ok else 'not the goal'})"

print("first 6 actions:", first_path[:6])
print("replay test:", replay(WAREHOUSE_MAP, first_path))

first 6 actions: ['Down', 'Down', 'Down', 'Down', 'Right', 'Down']
replay test: (False, 'move 6 (Down) collides at (6, 2)')


In [5]:
BLOCKED_MAP = WAREHOUSE_MAP.replace("...G#", "..#G#").replace("#.##....##########..#", "#.##....##########.##")
RAGGED_MAP = WAREHOUSE_MAP.replace("#.######.###.#.###..#", "#.######.###.#")
NO_START_MAP = WAREHOUSE_MAP.replace("S", ".")

for name, m in [("goal walled off", BLOCKED_MAP), ("ragged rows", RAGGED_MAP), ("no S", NO_START_MAP)]:
    try:
        print(f"{name:<16} -> returned {solve_warehouse(m)}")
    except Exception as e:
        print(f"{name:<16} -> {type(e).__name__}: {e}")

goal walled off  -> returned None
ragged rows      -> IndexError: list index out of range
no S             -> UnboundLocalError: cannot access local variable 'start' where it is not associated with a value


### What testing found

1. **The action labels are wrong.** This is the serious one. The code stores moves as `(dx, dy)`, an (x, y) convention, but adds them to `(row, col)`. So "Up" actually moves left, "Down" moves right, "Left" moves up and "Right" moves down. The *search* is correct: it explores all four neighbours and finds a genuinely shortest route, which is why the length (20) is right. But the printed instructions are wrong. The true route goes right along the top aisle, while the printout says `Down ×4, Right, Down, Down, …`. A vehicle obeying it would drive down the left-hand aisle and hit the bottom wall on move 6. The replay test catches this at once. Reading the printout would not, because a list of 20 plausible moves looks fine.
2. **Malformed maps crash with confusing errors.** A ragged row raises `IndexError`, because `cols` is taken from the first row only. A map with no `S` raises `UnboundLocalError` about a variable called `start`. Neither message says what is actually wrong with the map.
3. **Inefficiencies that do not change the answer.** Nodes are marked visited when they are *dequeued* rather than when they are *enqueued*, so the same cell can be queued several times. And every queue entry carries a full copy of its path (`path + [name]`), which costs memory proportional to the path length for every entry. Both are harmless on a 66-cell map but matter at scale (see Think About It).
4. **The no-path case works:** it returns `None`, and the program prints "No path exists".

So the answer to "did it work first time?" is: **it ran first time, but it was not correct.** The part a user would actually follow, the directions, was wrong.

### Improved prompt

The fix was to make the specification say what the first prompt left implicit:

> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem below.
>
> **Representation.** Parse the map into a 2-D grid indexed `grid[row][col]`, with row 0 at the top. Positions are `(row, col)` tuples. Define the actions exactly as `Up = (-1, 0)`, `Down = (+1, 0)`, `Left = (0, -1)`, `Right = (0, +1)` in `(d_row, d_col)`. Validate the map: all rows the same length, exactly one `S` and exactly one `G`, only the characters `#.SG`. Raise a `ValueError` with a clear message otherwise.
>
> **Agent structure.** Separate the environment (grid, `is_free`, `successors`, a `step` method that executes an action and raises on collision) from the agent (current state, goal, planner, plan executor).
>
> **Search.** Use breadth-first search with a FIFO queue. Mark cells as reached when they are *enqueued*, and store parent pointers instead of copying paths. Return the action list, the list of positions, and the number of nodes expanded. Return `None` if no path exists.
>
> **Output.** Print the action sequence, its length, and the map with the path drawn using `*`. Explain the chosen search algorithm and why it is appropriate.
>
> **Tests.** Include tests that (a) replay the returned actions from `S` using the definitions above and check that they reach `G` without collisions, (b) check the path is shortest by comparing with an independent BFS distance computed from `G`, (c) wall off `G` and check the no-path message, and (d) check that malformed maps raise `ValueError`.

The second attempt followed this specification. After inspection it is the implementation below, with A*, DFS and greedy best-first added as alternative planners for the comparisons.

In [6]:
class Warehouse:
    """The environment: a static, fully observable grid of free cells and obstacles."""

    def __init__(self, map_text):
        lines = [line for line in map_text.strip("\n").split("\n")]
        if not lines or len({len(line) for line in lines}) != 1:
            raise ValueError("map rows must all have the same length")
        bad = {ch for line in lines for ch in line} - set("#.SG")
        if bad:
            raise ValueError(f"unexpected characters in map: {sorted(bad)}")
        found = {ch: [(r, c) for r, line in enumerate(lines) for c, x in enumerate(line) if x == ch] for ch in "SG"}
        for ch in "SG":
            if len(found[ch]) != 1:
                raise ValueError(f"map must contain exactly one {ch!r}, found {len(found[ch])}")
        self.grid = lines
        self.n_rows, self.n_cols = len(lines), len(lines[0])
        self.start, self.goal = found["S"][0], found["G"][0]

    def is_free(self, pos):
        r, c = pos
        return 0 <= r < self.n_rows and 0 <= c < self.n_cols and self.grid[r][c] != "#"

    def successors(self, pos):
        # Transition model: the (action, next_position) pairs applicable in pos.
        for action, (dr, dc) in ACTIONS.items():
            nxt = (pos[0] + dr, pos[1] + dc)
            if self.is_free(nxt):
                yield action, nxt

    def step(self, pos, action):
        # Execute an action in the world; a collision is an error, not a silent no-op.
        dr, dc = ACTIONS[action]
        nxt = (pos[0] + dr, pos[1] + dc)
        if not self.is_free(nxt):
            raise RuntimeError(f"collision: {action} from {pos} hits an obstacle at {nxt}")
        return nxt

    def render(self, positions=()):
        on_path = set(positions) - {self.start, self.goal}
        return "\n".join("".join("*" if (r, c) in on_path else ch for c, ch in enumerate(line))
                         for r, line in enumerate(self.grid))

In [7]:
def reconstruct(parent, goal):
    actions, positions, node = [], [goal], goal
    while parent[node] is not None:
        prev, action = parent[node]
        actions.append(action)
        positions.append(prev)
        node = prev
    return actions[::-1], positions[::-1]


def bfs(env, start, goal):
    """Breadth-first search: expands positions in order of distance from start.

    Every move costs 1, so the first time BFS reaches the goal it has found a
    shortest path (optimal); it is complete on a finite grid; time and memory are
    O(number of free cells).
    """
    parent = {start: None}                 # reached set + parent pointers
    frontier = deque([start])
    expanded = 0
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return (*reconstruct(parent, goal), expanded)
        for action, nxt in env.successors(node):
            if nxt not in parent:          # mark reached on enqueue
                parent[nxt] = (node, action)
                frontier.append(nxt)
    return None, None, expanded


def dfs(env, start, goal):
    """Depth-first search: complete on a finite grid, but not optimal."""
    parent, frontier, expanded = {start: None}, [start], 0
    while frontier:
        node = frontier.pop()
        expanded += 1
        if node == goal:
            return (*reconstruct(parent, goal), expanded)
        for action, nxt in env.successors(node):
            if nxt not in parent:
                parent[nxt] = (node, action)
                frontier.append(nxt)
    return None, None, expanded


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def best_first(env, start, goal, weight_g):
    # Shared implementation of A* (f = g + h, weight_g=1) and greedy best-first (f = h, weight_g=0).
    g = {start: 0}
    parent = {start: None}
    counter = 0                            # FIFO tie-breaking for equal f
    frontier = [(manhattan(start, goal), counter, start)]
    closed, expanded = set(), 0
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1
        if node == goal:
            return (*reconstruct(parent, goal), expanded)
        for action, nxt in env.successors(node):
            ng = g[node] + 1
            if nxt not in g or ng < g[nxt]:
                g[nxt] = ng
                parent[nxt] = (node, action)
                counter += 1
                heapq.heappush(frontier, (weight_g * ng + manhattan(nxt, goal), counter, nxt))
    return None, None, expanded


def astar(env, start, goal):
    """A* with the Manhattan-distance heuristic (admissible and consistent here), so optimal."""
    return best_first(env, start, goal, weight_g=1)


def greedy(env, start, goal):
    """Greedy best-first: follows the heuristic only; fast but not optimal in general."""
    return best_first(env, start, goal, weight_g=0)

In [8]:
class GoalBasedAgent:
    """Goal-based agent: keeps its state and goal, plans with a search algorithm, executes the plan."""

    def __init__(self, env, planner=bfs):
        self.env = env
        self.planner = planner
        self.state = env.start             # percept: the vehicle knows where it is
        self.goal = env.goal
        self.plan = []
        self.stats = {}

    def goal_test(self):
        return self.state == self.goal

    def formulate_plan(self):
        actions, positions, expanded = self.planner(self.env, self.state, self.goal)
        self.stats = {"expanded": expanded, "planned_length": None if actions is None else len(actions)}
        self.plan = list(actions) if actions is not None else None
        return self.plan

    def act(self):
        # One agent-environment cycle: (re)plan if needed, take the next action, perceive the new state.
        if self.plan is None:
            return None
        if not self.plan:
            self.formulate_plan()
            if not self.plan:
                return None
        action = self.plan.pop(0)
        self.state = self.env.step(self.state, action)
        return action

    def run(self, max_steps=10_000):
        trajectory, actions = [self.state], []
        if self.formulate_plan() is None:
            return None, trajectory
        while not self.goal_test() and len(actions) < max_steps:
            a = self.act()
            if a is None:
                return None, trajectory
            actions.append(a)
            trajectory.append(self.state)
        return actions, trajectory

### Running the final agent

In [9]:
env = Warehouse(WAREHOUSE_MAP)
agent = GoalBasedAgent(env, planner=bfs)
actions, trajectory = agent.run()

def compress(actions):
    # "Right x4, Down x2, ..." is easier to read than 22 separate moves.
    out, i = [], 0
    while i < len(actions):
        j = i
        while j < len(actions) and actions[j] == actions[i]:
            j += 1
        out.append(f"{actions[i]} x{j - i}")
        i = j
    return ", ".join(out)

if actions is None:
    print("No collision-free path exists from S to G.")
else:
    print(f"Start {env.start} -> goal {env.goal}: {len(actions)} moves, {agent.stats['expanded']} nodes expanded\n")
    print("Actions:", compress(actions), "\n")
    print(env.render(trajectory))
    (OUT_DIR / "bfs_path.txt").write_text(
        f"BFS path, {len(actions)} moves\n{compress(actions)}\n\n{env.render(trajectory)}\n")

Start (1, 1) -> goal (1, 19): 20 moves, 59 nodes expanded

Actions: Right x3, Down x1, Right x3, Up x1, Right x12 

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


### Testing the final agent

In [10]:
def distances_from(env, source):
    # Independent BFS distance map, used as the ground truth for optimality.
    dist, q = {source: 0}, deque([source])
    while q:
        node = q.popleft()
        for _, nxt in env.successors(node):
            if nxt not in dist:
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return dist

def validate(env, actions):
    # Replay the plan in the environment: every step must be legal and the last must reach G.
    pos = env.start
    for a in actions:
        pos = env.step(pos, a)            # raises on collision
    return pos == env.goal

# (a) replay: collision-free and reaches G
assert validate(env, actions), "plan does not reach G"
assert replay(WAREHOUSE_MAP, actions)[0]
# (b) optimal: matches the true shortest distance computed backwards from G
shortest = distances_from(env, env.goal)[env.start]
assert len(actions) == shortest, (len(actions), shortest)
# (c) no path: goal walled off
blocked = GoalBasedAgent(Warehouse(BLOCKED_MAP)).run()
assert blocked[0] is None
# (d) malformed maps give clear errors
for bad in [RAGGED_MAP, NO_START_MAP, WAREHOUSE_MAP.replace(".", "x", 1), WAREHOUSE_MAP.replace("#.##", "#S##")]:
    try:
        Warehouse(bad)
        raise AssertionError("malformed map accepted")
    except ValueError as e:
        print("ValueError:", e)
# (e) S == G neighbour and trivial cases
tiny = Warehouse("####\n#SG#\n####")
assert bfs(tiny, tiny.start, tiny.goal)[0] == ["Right"]

print(f"\nAll tests passed: plan is collision-free, reaches G, and its length {len(actions)} equals the true shortest distance {shortest}.")
print("Goal walled off ->", "No collision-free path exists from S to G.")

ValueError: map rows must all have the same length
ValueError: map must contain exactly one 'S', found 0
ValueError: unexpected characters in map: ['x']
ValueError: map must contain exactly one 'S', found 5

All tests passed: plan is collision-free, reaches G, and its length 20 equals the true shortest distance 20.
Goal walled off -> No collision-free path exists from S to G.


### Comparing search strategies on this map

The LLM chose BFS. To judge that choice, the other standard strategies were run on the same map:

In [11]:
def count_shortest_paths(env):
    dist_s, dist_g = distances_from(env, env.start), distances_from(env, env.goal)
    L = dist_s[env.goal]
    ways = {env.start: 1}
    for node in sorted(dist_s, key=dist_s.get):
        for _, nxt in env.successors(node):
            if dist_s.get(nxt) == dist_s[node] + 1:
                ways[nxt] = ways.get(nxt, 0) + ways[node]
    return ways[env.goal]

print(f"{'strategy':<24}{'path length':>12}{'nodes expanded':>16}{'optimal?':>10}")
for name, planner in [("BFS", bfs), ("A* (Manhattan)", astar), ("Greedy best-first", greedy), ("DFS", dfs)]:
    acts, pos, expanded = planner(env, env.start, env.goal)
    assert validate(env, acts)
    print(f"{name:<24}{len(acts):>12}{expanded:>16}{'yes' if len(acts) == shortest else 'no':>10}")
print(f"\nThere are {count_shortest_paths(env)} different shortest paths of length {shortest}.")
print("\nDFS path:\n" + env.render(dfs(env, env.start, env.goal)[1]))

strategy                 path length  nodes expanded  optimal?
BFS                               20              59       yes
A* (Manhattan)                    20              23       yes
Greedy best-first                 20              21       yes
DFS                               22              34        no

There are 2 different shortest paths of length 20.

DFS path:
#####################
#S****#............G#
#.##.***##########.*#
#....##*************#
#.######.###.#.###..#
#........#..........#
#####################


On this 66-cell map every strategy is instant:
- **BFS and A\*** both return an optimal 20-move path.
- **A\*** expands fewer nodes, because the Manhattan heuristic steers it towards `G`.
- **Greedy best-first** follows the heuristic straight at `G`. It happens to find an optimal path here too, but it has no guarantee of doing so.
- **DFS** returns a valid but much longer, winding path.

The problem is too small to separate the strategies on cost. The next section makes it bigger.

## Think About It — what if the warehouse becomes twice as large?

First, the literal case: the same layout tiled 2 × 2 (twice as many rows and twice as many columns, so four times the area), with a cross-aisle between the four copies. `S` stays top-left and `G` moves to the bottom-right corner:

In [12]:
def tile_2x2(map_text):
    inner = [row[1:-1] for row in map_text.strip("\n").split("\n")[1:-1]]
    inner = [row.replace("S", ".").replace("G", ".") for row in inner]
    big = [a + "." + a for a in inner]                  # a cross-aisle between the copies
    big = big + ["." * len(big[0])] + big
    big[0] = "S" + big[0][1:]
    big[-1] = big[-1][:-1] + "G"
    width = len(big[0]) + 2
    return "\n".join(["#" * width] + ["#" + row + "#" for row in big] + ["#" * width])

BIG_MAP = tile_2x2(WAREHOUSE_MAP)
big = Warehouse(BIG_MAP)
print(f"{big.n_rows} x {big.n_cols} grid, {sum(big.is_free((r, c)) for r in range(big.n_rows) for c in range(big.n_cols))} free cells\n")
print(f"{'strategy':<20}{'length':>8}{'expanded':>10}")
for name, planner in [("BFS", bfs), ("A*", astar), ("Greedy", greedy), ("DFS", dfs)]:
    acts, pos, expanded = planner(big, big.start, big.goal)
    print(f"{name:<20}{len(acts) if acts else '-':>8}{expanded:>10}")
print("\nA* path:\n" + big.render(astar(big, big.start, big.goal)[1]))

13 x 41 grid, 313 free cells

strategy              length  expanded
BFS                       48       313
A*                        48       237
Greedy                    56        57
DFS                      130       131

A* path:
#########################################
#S....#...................#.............#
#*##....##########....##....##########..#
#*...##..................##.............#
#*######.###.#.###....######.###.#.###..#
#*.......#...................#..........#
#*******................................#
#.....#******************.#.............#
#.##....##########....##****##########..#
#....##..................##******.......#
#.######.###.#.###....######.###*#.###..#
#........#...................#..*******G#
#########################################


To see the trend, random warehouses were generated at increasing sizes. Each has rows of shelving with gaps (aisles), plus some random clutter, and the width doubles each time. For each size the table reports the mean over 5 random layouts:

In [13]:
def random_warehouse(n_rows, n_cols, seed, gap=0.25, clutter=0.08):
    # Shelving every other row with random aisle gaps, plus random clutter; S top-left, G bottom-right.
    rng = random.Random(seed)
    g = [["#"] * n_cols] + [["#"] + ["."] * (n_cols - 2) + ["#"] for _ in range(n_rows - 2)] + [["#"] * n_cols]
    for r in range(2, n_rows - 2, 2):
        for c in range(1, n_cols - 1):
            if rng.random() > gap:
                g[r][c] = "#"
    for r in range(1, n_rows - 1):
        for c in range(1, n_cols - 1):
            if g[r][c] == "." and rng.random() < clutter:
                g[r][c] = "#"
    g[1][1], g[n_rows - 2][n_cols - 2] = "S", "G"
    return "\n".join("".join(row) for row in g)

def compare(env, repeats=3):
    out = {}
    for name, planner in [("BFS", bfs), ("A*", astar), ("Greedy", greedy), ("DFS", dfs)]:
        t0 = time.perf_counter()
        for _ in range(repeats):
            acts, _, expanded = planner(env, env.start, env.goal)
        out[name] = (None if acts is None else len(acts), expanded, (time.perf_counter() - t0) / repeats * 1000)
    return out

sizes = [(7, 21), (14, 42), (28, 84), (56, 168), (112, 336)]
rows_out = []
for n_rows, n_cols in sizes:
    agg = {k: [0, 0, 0.0, 0] for k in ["BFS", "A*", "Greedy", "DFS"]}
    solved = 0
    for seed in range(40):
        env_r = Warehouse(random_warehouse(n_rows, n_cols, seed))
        res = compare(env_r)
        if res["BFS"][0] is None:
            continue                          # unsolvable layout; skip it
        solved += 1
        for k, (L, e, t) in res.items():
            agg[k][0] += L; agg[k][1] += e; agg[k][2] += t
        if solved == 5:
            break
    free = sum(env_r.is_free((r, c)) for r in range(n_rows) for c in range(n_cols))
    rows_out.append((n_rows, n_cols, {k: [v[0] / solved, v[1] / solved, v[2] / solved] for k, v in agg.items()}))

print(f"{'grid':>9} | {'BFS len':>7} {'exp':>6} {'ms':>6} | {'A* len':>6} {'exp':>6} {'ms':>6} | "
      f"{'Greedy len':>10} {'exp':>5} | {'DFS len':>7} {'exp':>6}")
for n_rows, n_cols, a in rows_out:
    print(f"{n_rows:>3}x{n_cols:<5} | {a['BFS'][0]:>7.1f} {a['BFS'][1]:>6.0f} {a['BFS'][2]:>6.2f} | "
          f"{a['A*'][0]:>6.1f} {a['A*'][1]:>6.0f} {a['A*'][2]:>6.2f} | "
          f"{a['Greedy'][0]:>10.1f} {a['Greedy'][1]:>5.0f} | {a['DFS'][0]:>7.1f} {a['DFS'][1]:>6.0f}")

with open(OUT_DIR / "scaling_results.txt", "w") as f:
    f.write("mean over 5 solvable random warehouses per size\n")
    for n_rows, n_cols, a in rows_out:
        f.write(f"{n_rows}x{n_cols}: " + ", ".join(f"{k}: len {v[0]:.1f}, expanded {v[1]:.0f}, {v[2]:.2f} ms" for k, v in a.items()) + "\n")

     grid | BFS len    exp     ms | A* len    exp     ms | Greedy len   exp | DFS len    exp
  7x21    |    23.6     64   0.06 |   23.6     56   0.08 |       28.0    33 |    26.8     32
 14x42    |    50.4    286   0.27 |   50.4    167   0.25 |       53.6    61 |    66.0    114
 28x84    |   107.6   1209   1.15 |  107.6    423   0.64 |      115.6   128 |   223.2    620
 56x168   |   222.8   5130   5.17 |  222.8   1623   2.57 |      262.0   364 |   534.4   1482
112x336   |   453.2  20719  21.87 |  453.2   5139   9.15 |      510.0   698 |  2139.6   7050


### Answer: would the same search strategy still be appropriate?

**At twice the size, yes.** BFS is still complete and still optimal, because every move still costs 1, and its cost grows only linearly with the number of free cells, $O(|V| + |E|)$. Doubling both dimensions quadruples the cells, so BFS does roughly 4× the work. That is still milliseconds even for the largest warehouse in the table. The algorithm's *correctness* does not depend on size at all.

The measurements confirm this:
- **Tiled 13 × 41 warehouse** (the literal "twice as large"): BFS expands all 313 reachable cells to find the optimal 48-move path, and A* finds the same 48 moves after expanding 237.
- **Random warehouses:** BFS's work grows roughly 4× per doubling of the side length (64 → 286 → 1209 → 5130 → 20,719 nodes). At 112 × 336 it still takes only about 20 ms.

**As the warehouse keeps growing, BFS becomes the wrong choice for efficiency, though not for correctness:**
- **BFS is blind.** It expands cells in every direction, including away from the goal, so it ends up exploring nearly the whole reachable warehouse on every query. **A\*** with the Manhattan heuristic returns the *same* optimal path length but expands only a fraction of the nodes in the table, and the gap widens as the grid grows. A* is the natural upgrade.
  At 112 × 336, A* expands about 5,100 nodes against BFS's 20,700 (4× fewer) for the identical 453-move path.
- **Greedy best-first** expands the fewest nodes of all (about 700 at the largest size), but gives up the guarantee of a shortest path. On the tiled map its path is 56 moves instead of 48, and on the random maps its paths are 6–18% longer than optimal, depending on size. **DFS** is never a good choice here. Its paths get far longer than optimal as the map grows: 130 instead of 48 on the tiled map, and 2,140 instead of 453 on the largest random map.

**Additional difficulties that arise with a larger warehouse:**
1. **Computation and memory.** The frontier and reached set grow with the area (quadratically in the side length). The first LLM version, which copies the whole path into every queue entry, would use memory proportional to (number of cells) × (path length). Parent pointers matter.
2. **Many queries.** A real warehouse plans thousands of routes. Techniques that reuse work start to pay off: precomputed distance maps to each dispatch bay, hierarchical planning over aisles and zones, or incremental replanning algorithms (D\* Lite, LPA\*).
3. **Ties and path quality.** With many equal-length shortest paths (the small map alone has several), the planner should break ties in favour of fewer turns, wider aisles or lower traffic. That makes the cost non-uniform, so BFS no longer applies and uniform-cost search or A* with real costs is needed.
4. **The environment stops being static and single-agent.** More floor space means more vehicles, people and pallets. Paths must avoid other vehicles (multi-agent path finding, time-expanded search), and the agent must detect blocked cells and replan while moving. The `step` method raising on collision, followed by a call to `formulate_plan()` from the current state, is the hook for that.
5. **Partial observability and scale of the map itself.** The vehicle may no longer know the whole map. It has to sense locally and keep its map up to date, and planning with an incomplete map becomes a harder problem.
6. **Validation.** Hand-checking a route stops being possible. Automated tests (replay, independent shortest-distance checks) are what make the answer trustworthy.

### Replanning: the goal-based design pays off

A small demonstration of difficulty 4. A pallet is dropped in the aisle while the vehicle is en route. The move fails, the agent perceives that, updates its map, and replans from where it is:

In [14]:
env2 = Warehouse(WAREHOUSE_MAP)
agent2 = GoalBasedAgent(env2)
agent2.formulate_plan()
original_plan = list(agent2.plan)
trajectory2 = [agent2.state]
blocked_cell = None
while not agent2.goal_test():
    if len(trajectory2) == 8 and blocked_cell is None:
        # Drop a pallet on the cell the agent is about to enter.
        dr, dc = ACTIONS[agent2.plan[0]]
        blocked_cell = (agent2.state[0] + dr, agent2.state[1] + dc)
        row = list(env2.grid[blocked_cell[0]]); row[blocked_cell[1]] = "#"; env2.grid[blocked_cell[0]] = "".join(row)
    try:
        agent2.act()
    except RuntimeError as e:
        print("perceived:", e)
        agent2.formulate_plan()                   # replan from the current state
        print(f"replanned from {agent2.state}: {agent2.stats['planned_length']} moves remaining")
        continue
    trajectory2.append(agent2.state)

print(f"\noriginal plan: {len(original_plan)} moves; executed: {len(trajectory2) - 1} moves; reached G: {agent2.goal_test()}")
print(env2.render(trajectory2))

perceived: collision: Up from (2, 7) hits an obstacle at (1, 7)
replanned from (2, 7): 15 moves remaining

original plan: 20 moves; executed: 22 moves; reached G: True
#####################
#S***.##..........*G#
#.##****##########*.#
#....##************.#
#.######.###.#.###..#
#........#..........#
#####################


## Task 3 — Questions

**1. Did the LLM generate a working program on the first attempt?**
It generated a program that **ran** on the first attempt: no syntax or runtime errors on the given map. It found a path of the correct, shortest length (20 moves) and handled the no-path case. But it was **not correct**. Its `(dx, dy)` move table was applied to `(row, col)` positions, so every printed action was mislabelled ("Up" actually moved left, "Down" moved right). A vehicle following its instructions would crash into the wall on move 6. It also crashed with unhelpful errors on malformed maps. The replay test exposed the main bug at once, even though the printed output looked plausible. "Runs" and "works" are different claims.

**2. How can you improve the prompt?**
Make the implicit parts of the specification explicit and ask for the checks up front (the full improved prompt is above):
- **Fix the conventions:** `grid[row][col]`, positions as `(row, col)`, and the exact `(d_row, d_col)` of each action. This removes the (x, y) vs (row, col) ambiguity that caused the main bug.
- **Specify input validation:** rectangular rows, exactly one `S` and one `G`, allowed characters, and a clear `ValueError` otherwise.
- **Specify the agent structure:** separate environment, state, goal, planner and executor, matching the Task 2 design, so the code is recognisably a goal-based agent rather than just a function.
- **Specify implementation details that matter:** mark cells reached when enqueued, store parent pointers rather than copies of paths, report nodes expanded.
- **Specify the output format:** the action list, its length and a drawn map, which makes errors visible to a human.
- **Ask for tests:** replay the plan, check optimality against an independent computation, check the no-path and malformed-input cases.

Iterating with the LLM also works better when you give it the *failing test*: "replaying your actions from S collides on move 1". A vague "it's wrong" gives it nothing to work with.

**3. What search algorithm did the LLM choose?**
**Breadth-first search (BFS)**, with a FIFO queue and a visited set, in both attempts.

**4. Why do you think the LLM selected this algorithm?**
- **It is the textbook-correct choice for this problem.** All moves cost the same, so BFS is complete and returns a shortest path. The state space is small and fully known, and BFS is simple to implement and to explain. It satisfies the prompt's requirements (collision-free path, "no path" message, justification) with the least machinery.
- **It is the most common pattern in the LLM's training data.** "Grid maze shortest path in Python" is overwhelmingly answered with BFS in tutorials, textbooks and programming-contest solutions. An LLM predicts the most typical solution for the prompt it is given, and the lab's prompt did not mention costs, heuristics, map size or performance, which are the things that would point towards A*.

The choice is appropriate here, but it was not made from an analysis of *this* warehouse's size or future requirements. Deciding whether A* or replanning is needed (Think About It) is still the engineer's job.